# Nodes and running

A node is an ordinary typed Python function or a small class. Linking nodes
builds a graph, and `app.run()` runs it and hands back every node's outputs. This
notebook builds the sum `(3 + 4) x 3`, then a class node, then shows what
happens when a node cannot run.

In [1]:
# Standard library
import math
from typing import override

In [2]:
# Local libraries
import element_51
from element_51.edits import EditError, SetValue

In [3]:
# What the graphs below should give
SUM = 7.0
PRODUCT = 21.0
DOUBLED = 6.0

## Functions become nodes

Each parameter becomes an input port and the return type becomes the output
port. The names of the ports are the parameter names and `out`.

In [4]:
@element_51.node
def three() -> float:
    return 3.0


@element_51.node
def four() -> float:
    return 4.0


@element_51.node
def add(a: float, b: float) -> float:
    return a + b


@element_51.node
def multiply(a: float, b: float) -> float:
    return a * b

In [5]:
app = element_51.App([three, four, add, multiply])
n_three = app.add_node(three, x=60.0, y=80.0)
n_four = app.add_node(four, x=60.0, y=220.0)
n_add = app.add_node(add, x=320.0, y=120.0)
n_multiply = app.add_node(multiply, x=580.0, y=120.0)
app.add_link(n_three, "out", n_add, "a")
app.add_link(n_four, "out", n_add, "b")
app.add_link(n_add, "out", n_multiply, "a")
app.add_link(n_three, "out", n_multiply, "b")
print(f"{len(app.snapshot().graph.nodes)} nodes")

4 nodes


Run it. The result maps each node id to its outputs by port name.

In [6]:
results = app.run()
for node_id, outputs in results.items():
    print(node_id, outputs)
total = results[n_add]["out"]
product = results[n_multiply]["out"]
assert isinstance(total, float)
assert isinstance(product, float)
assert math.isclose(total, SUM)
assert math.isclose(product, PRODUCT)

n1 {'out': 3.0}
n2 {'out': 4.0}
n3 {'out': 7.0}
n4 {'out': 21.0}


## A class is a node with values you set

A class node holds its inputs as fields with defaults, and `run` returns the
output. An unlinked input keeps the value typed into the node.

In [7]:
class Double(element_51.Node):
    value: float = 0.0

    @override
    def run(self) -> float:
        return self.value * 2

In [8]:
small = element_51.App([three, Double])
n_source = small.add_node(three, x=60.0, y=80.0)
n_double = small.add_node(Double, x=320.0, y=80.0)
small.add_link(n_source, "out", n_double, "value")
print(small.run()[n_double])
doubled = small.run()[n_double]["out"]
assert isinstance(doubled, float)
assert math.isclose(doubled, DOUBLED)

{'out': 6.0}


## A node that cannot run

`app.execute()` reports what happened to every node and raises nothing. `app.run()`
raises the node's own error, with a note that names the node. A node whose input
has no link and no value cannot run, and the nodes after it do not run either.

In [9]:
@element_51.node
def split(amount: float, people: float) -> float:
    return amount / people


broken = element_51.App([three, split])
b_three = broken.add_node(three, x=60.0, y=80.0)
b_split = broken.add_node(split, x=320.0, y=80.0)
broken.add_link(b_three, "out", b_split, "amount")
for outcome in broken.execute().outcomes:
    detail = outcome.error if outcome.status == "error" else outcome.outputs
    print(outcome.node, outcome.status, detail)

n1 ok {'out': 3.0}
n2 error input 'people' has no value


In [10]:
raised = False
try:
    broken.run()
except ValueError as error:
    raised = True
    print("raised:", error)
    print("note:", error.__notes__[0])
assert raised

raised: input 'people' has no value
note: in node n2 (split)


## A link between ports of different types is refused

In [11]:
@element_51.node
def label(text: str) -> str:
    return text


typed = element_51.App([three, label])
t_three = typed.add_node(three, x=60.0, y=80.0)
t_label = typed.add_node(label, x=320.0, y=80.0)
refused = False
try:
    typed.add_link(t_three, "out", t_label, "text")
except EditError as error:
    refused = True
    print("refused:", error)
assert refused

refused: cannot link float to str


## Default nodes

Every app offers `Plot` without being given it, in the `Default nodes` folder of the
menu. The nodes you list go under `User defined nodes`. `defaults=False` turns the
default nodes off, and the menu shows your nodes under the folders they declare.

In [12]:
def menu(app: element_51.App) -> dict[str, str]:
    """Read the menu path of each node type the page is told about.

    Returns
    -------
    dict[str, str]
        Node type name to category.
    """
    return {t.name: t.category for t in app.snapshot().node_types}


with_defaults = menu(element_51.App([three, label]))
plain = menu(element_51.App([three, label], defaults=False))
print(with_defaults)
print(plain)
assert with_defaults["Plot"] == "Default nodes/Output nodes"
assert with_defaults["three"] == "User defined nodes"
assert "Plot" not in plain
assert not plain["three"]

{'Float': 'Default nodes/Input nodes', 'Integer': 'Default nodes/Input nodes', 'Text': 'Default nodes/Input nodes', 'Flag': 'Default nodes/Input nodes', 'Slider': 'Default nodes/Input nodes', 'Plot': 'Default nodes/Output nodes', 'three': 'User defined nodes', 'label': 'User defined nodes'}
{'three': '', 'label': ''}


## Input nodes

`Float`, `Integer`, `Text` and `Flag` hold a value you set and pass it on, so one value
can feed any number of inputs. `Slider` is a `Float` with a minimum and a maximum. A
value outside them, or an empty range, is an error on the card, not a silent clamp.

In [13]:
inputs = element_51.App([add])
n_float = inputs.add_node(element_51.Float, x=0.0, y=0.0)
n_total = inputs.add_node(add, x=300.0, y=0.0)
inputs.apply(SetValue(node=n_float, port="value", value=3.0))
inputs.add_link(n_float, "out", n_total, "a")
inputs.add_link(n_float, "out", n_total, "b")
print(inputs.run()[n_total])
assert inputs.run()[n_total] == {"out": 6.0}

{'out': 6.0}


In [14]:
slider = inputs.add_node(element_51.Slider, x=0.0, y=200.0)
inputs.apply(SetValue(node=slider, port="value", value=11.0))
try:
    inputs.run()
except ValueError as error:
    print("raised:", error)
    outside = True
else:
    outside = False
assert outside

raised: the value 11.0 must be between -10.0 and 10.0
